# Baseline: LSTM for Long-Term Time Series Forecasting

## 0. Get Setup

In [1]:
# Clone the github repo and download dependencies
!git clone https://github.com/chunyagi/PatchTST.git

Cloning into 'PatchTST'...
remote: Enumerating objects: 438, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 438 (delta 19), reused 17 (delta 16), pack-reused 406 (from 2)
Receiving objects: 100% (438/438), 12.97 MiB | 18.73 MiB/s, done.
Resolving deltas: 100% (202/202), done.


## 1. Download Data

In [2]:
# Download data from google drive with gdown
try:
    import gdown
    print(f"[INFO] gdown version: {gdown.__version__}")
except:
    print("[INFO] Couldn't find gdown, installing it...")
    !pip install -q gdown
    import gdown
    print(f"[INFO] gdown version: {gdown.__version__}")

# ETTh1/m1
!mkdir -p /data/datasets/public/ETDataset/ETT-small
!gdown 1vOClm_t4RgUf8nqherpTfNnB8rrYq14Q -O /data/datasets/public/ETDataset/ETT-small/
!gdown 1B7VcTWdIfPl3g17zKXATKF9XQJtNHTtl -O /data/datasets/public/ETDataset/ETT-small/

# Weather
!mkdir -p /data/datasets/public/weather/
!gdown 1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy -O /data/datasets/public/weather/

# Electricity
!mkdir -p /data/datasets/public/electricity/
!gdown 1jinfTAApPyuyvW1P1hUDpI3rl0Jq8in1 -O /data/datasets/public/electricity/

[INFO] gdown version: 5.2.2
Downloading...
From: https://drive.google.com/uc?id=1vOClm_t4RgUf8nqherpTfNnB8rrYq14Q
To: /data/datasets/public/ETDataset/ETT-small/ETTh1.csv
100% 2.59M/2.59M [00:00<00:00, 238MB/s]
Downloading...
From: https://drive.google.com/uc?id=1B7VcTWdIfPl3g17zKXATKF9XQJtNHTtl
To: /data/datasets/public/ETDataset/ETT-small/ETTm1.csv
100% 10.4M/10.4M [00:00<00:00, 44.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Tc7GeVN7DLEl-RAs-JVwG9yFMf--S8dy
To: /data/datasets/public/weather/weather.csv
100% 7.24M/7.24M [00:00<00:00, 22.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jinfTAApPyuyvW1P1hUDpI3rl0Jq8in1
To: /data/datasets/public/electricity/electricity.csv
100% 95.6M/95.6M [00:01<00:00, 90.8MB/s]


## 2. Model Class

In [3]:
# Create directory
!mkdir -p /content/lstm_baseline/models

In [4]:
%%writefile /content/lstm_baseline/models/lstm.py
import torch
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, pred_len=96, dropout=0.2):
        super(LSTM, self).__init__()
        self.input_size = input_size # number of features
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.pred_len = pred_len
        self.dropout = dropout

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, input_size * pred_len)

    def forward(self, x):
        # x: (batch_size, seq_len, input_size)
        lstm_out, _ = self.lstm(x)  # (batch_size, seq_len, hidden_size)

        # Use last hidden state
        last_hidden = lstm_out[:, -1, :]  # (batch_size, hidden_size)

        # Apply dropout
        last_hidden = self.dropout(last_hidden)

        # Predict all future steps
        output = self.fc(last_hidden)  # (batch_size, input_size * pred_len)
        output = output.view(-1, self.pred_len, self.input_size)  # (batch_size, pred_len, input_size)

        return output

Writing /content/lstm_baseline/models/lstm.py


## 3. Helper Functions

In [5]:
%%writefile /content/lstm_baseline/helpers.py

import numpy as np
import torch
from data_provider.data_factory import data_provider
from models.lstm import LSTM

def set_device():
  """Set up device agnostic code."""
  if torch.cuda.is_available():
    device = "cuda"
  elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = "mps"
  else:
    device = "cpu"
  return device

device = set_device()
print(f"Using device: {device}")


# Model evaluation
def vali(model, val_loader, criterion, device):
    """Evaluate model on validation/test dataloader."""
    model.eval()
    total_loss = 0
    total_mae = 0
    with torch.no_grad():
        for batch_x, batch_y, _, _ in val_loader:
            batch_x, batch_y = batch_x.float().to(device), batch_y.float().to(device)
            pred = model(batch_x)
            loss = criterion(pred, batch_y)
            mae = torch.mean(torch.abs(pred - batch_y))

            total_loss += loss.item()
            total_mae += mae.item()

    return total_loss / len(val_loader), total_mae / len(val_loader)


def _get_data(args, flag):
    """Get dataset and dataloader."""
    data_set, data_loader = data_provider(args, flag)
    return data_set, data_loader


def _build_model(args):
    """Build LSTM model"""
    model = LSTM(
        input_size=args.enc_in,
        hidden_size=args.hidden_size,
        num_layers=args.num_layers,
        pred_len=args.pred_len,
        dropout=args.dropout
    )
    return model

Writing /content/lstm_baseline/helpers.py


## 4. Training Script

In [6]:
%%writefile /content/lstm_baseline/train.py

import argparse
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import time
import sys

# Add PatchTST path for data utilities
sys.path.append('/content/PatchTST/PatchTST_supervised/')

from models.lstm import LSTM
from helpers import set_device, vali, _get_data, _build_model
from utils.tools import adjust_learning_rate, EarlyStopping


def train(args, setting):
    """Training loop for LSTM."""
    # get available GPU device
    device = set_device()

    # set random seed
    fix_seed = args.random_seed
    random.seed(fix_seed)
    torch.manual_seed(fix_seed)
    np.random.seed(fix_seed)

    # get datasets and dataloaders
    train_data, train_loader = _get_data(args, flag='train')
    vali_data, vali_loader = _get_data(args, flag='val')
    test_data, test_loader = _get_data(args, flag='test')
    # print(f'train data: {train_data[0][0]}')
    # print(f'train data: {train_data[0][0].shape}')
    # print(f'test data: {test_data[0][0]}')
    # print(f'test data: {test_data[0][0].shape}')

    # create save path
    path = os.path.join(args.checkpoints, setting)
    if not os.path.exists(path):
        os.makedirs(path)

    time_now = time.time()

    train_steps = len(train_loader)

    # early stopping
    early_stopping = EarlyStopping(patience=args.patience, verbose=True)

    # build model
    model = _build_model(args).to(device)

    # setup optimizer and loss function
    model_optim = torch.optim.Adam(model.parameters(), lr=args.learning_rate)
    criterion = nn.MSELoss()

    # training loop
    for epoch in range(args.train_epochs):
        iter_count = 0
        train_loss = []

        model.train()
        epoch_time = time.time()
        for i, (batch_x, batch_y, batch_x_mark, batch_y_mark) in enumerate(train_loader):
            iter_count += 1
            model_optim.zero_grad()

            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            train_loss.append(loss.item())

            # if (i + 1) % 10 == 0:
            #     print("\titers: {0}, epoch: {1} | loss: {2:.7f}".format(i + 1, epoch + 1, loss.item()))
            #     speed = (time.time() - time_now) / iter_count
            #     left_time = speed * ((args.train_epochs - epoch) * train_steps - i)
            #     print('\tspeed: {:.4f}s/iter; left time: {:.4f}s'.format(speed, left_time))
            #     iter_count = 0
            #     time_now = time.time()

            loss.backward()
            model_optim.step()

        # validation
        print("Epoch: {} cost time: {}".format(epoch + 1, time.time() - epoch_time))
        train_loss = np.average(train_loss)
        vali_loss, vali_mae = vali(model, vali_loader, criterion, device)
        test_loss, test_mae = vali(model, test_loader, criterion, device)

        # print results
        print("Epoch: {0}, Steps: {1} | Train Loss: {2:.7f} Vali Loss: {3:.7f} Test Loss: {4:.7f}".format(
            epoch + 1, train_steps, train_loss, vali_loss, test_loss))
        early_stopping(vali_loss, model, path)
        if early_stopping.early_stop:
            print("Early stopping")
            break

        # adjust learning rate after each epoch
        adjust_learning_rate(model_optim, None, epoch + 1, args)


def test(args, setting):
    """Evaluate trained model on test set."""
    device = set_device()
    criterion = nn.MSELoss()

    test_data, test_loader = _get_data(args, flag='test')
    # print(f'len of test_data: {len(test_data)}')

    # build model
    model = _build_model(args).to(device)

    # load saved model
    print('loading model')
    model.load_state_dict(torch.load(os.path.join(args.checkpoints, setting, 'checkpoint.pth')))

    # evaluate model
    mse, mae = vali(model, test_loader, criterion, device)
    print(f'Test MSE: {mse}, MAE: {mae}')

    # save results
    results_df = pd.DataFrame({'mse': [mse], 'mae': [mae]})
    results_df.to_csv(os.path.join(args.checkpoints, setting, 'result.csv'), index=False)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='LSTM Baseline for Long-term Time Series Forecasting')

    # random seed
    parser.add_argument('--random_seed', type=int, default=2021, help='random seed')

    # basic config
    parser.add_argument('--model_id', type=str, default='lstm_test', help='model id')

    # data loader
    parser.add_argument('--data', type=str, default='ETTh1', help='dataset type')
    parser.add_argument('--root_path', type=str, default='/data/datasets/public/ETDataset/ETT-small/', help='root path of the data file')
    parser.add_argument('--data_path', type=str, default='ETTh1.csv', help='data file')
    parser.add_argument('--features', type=str, default='M',
                            help='forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate')
    parser.add_argument('--target', type=str, default='OT', help='target feature in S or MS task')
    parser.add_argument('--freq', type=str, default='h',
                            help='freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h')
    parser.add_argument('--percent', type=int, default=100, help='percentage of training data to use')
    parser.add_argument('--embed', type=str, default='timeF',
                            help='time features encoding, options:[timeF, fixed, learned]')
    parser.add_argument('--checkpoints', type=str, default='./checkpoints/', help='location of model checkpoints')

    # forecasting task
    parser.add_argument('--seq_len', type=int, default=512, help='input sequence length')
    parser.add_argument('--label_len', type=int, default=0, help='start token length')
    parser.add_argument('--pred_len', type=int, default=96, help='prediction sequence length')

    # lstm
    parser.add_argument('--enc_in', type=int, default=7, help='encoder input size (number of features)')
    parser.add_argument('--hidden_size', type=int, default=8, help='LSTM hidden size')
    parser.add_argument('--num_layers', type=int, default=1, help='number of LSTM layers')
    parser.add_argument('--dropout', type=float, default=0.1, help='dropout rate')

    # optimization
    parser.add_argument('--num_workers', type=int, default=2, help='data loader num workers')
    parser.add_argument('--itr', type=int, default=1, help='experiments times')
    parser.add_argument('--train_epochs', type=int, default=50, help='train epochs')
    parser.add_argument('--batch_size', type=int, default=128, help='batch size of train input data')
    parser.add_argument('--patience', type=int, default=7, help='early stopping patience')
    parser.add_argument('--learning_rate', type=float, default=1e-2, help='optimizer learning rate')
    parser.add_argument('--lradj', type=str, default='type3', help='adjust learning rate')

    args = parser.parse_args()
    print('Args in experiment:')
    print(args)

    setting = f'{args.model_id}_{args.data}_sl{args.seq_len}_pl{args.pred_len}'

    # training
    print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
    train(args, setting)

    # testing
    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    test(args, setting)

Writing /content/lstm_baseline/train.py


In [7]:
%cd lstm_baseline

/content/lstm_baseline


## 5. Train Model on ETTh1

### 5.1 Horizon 96

In [ ]:
!python train.py  --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_1 --random_seed 2019
!python train.py  --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_2 --random_seed 2020
!python train.py  --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_3 --random_seed 2021
!python train.py  --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_4 --random_seed 2022
!python train.py  --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_96_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=96, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_96_1_ETTh1_sl512_pl96>>>>>>>>>>>>>>>>>>>>>>>>>>
train 8033
val 2785
test 2785
Epoch: 1 cost time: 0.359591007232666
Epoch: 1, Steps: 62 | Train Loss: 0.7242233 Vali Loss: 1.3946003 Test Loss: 1.0767409
Validation loss decreased (inf --> 1.394600).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.25155162811279297
Epoch: 2, Steps: 62 | Train Loss: 0.5404895 Vali Loss: 1.2777215 Test Loss: 1.0257701
Validation loss decreased (1.394600 --> 1.277721).  Saving model .

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.8865800145126524, 0.9033145691667285, 0.9261830960001264, 0.8774100797516959, 0.919237957114265])
scores_mae = np.array([0.7224727556819007, 0.7225785965011233, 0.7752173684892201, 0.7289137102308727, 0.7565121139798846])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.903 ± 0.019 | MAE: 0.741 ± 0.021


### 5.2 Horizon 192

In [ ]:
!python train.py  --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_1 --random_seed 2019
!python train.py  --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_2 --random_seed 2020
!python train.py  --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_3 --random_seed 2021
!python train.py  --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_4 --random_seed 2022
!python train.py  --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_192_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=192, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_192_1_ETTh1_sl512_pl192>>>>>>>>>>>>>>>>>>>>>>>>>>
train 7937
val 2689
test 2689
Epoch: 1 cost time: 0.3708665370941162
Epoch: 1, Steps: 62 | Train Loss: 0.7332814 Vali Loss: 1.4933287 Test Loss: 1.1028439
Validation loss decreased (inf --> 1.493329).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.27433252334594727
Epoch: 2, Steps: 62 | Train Loss: 0.5644702 Vali Loss: 1.4034423 Test Loss: 1.0400550
Validation loss decreased (1.493329 --> 1.403442).  Saving mo

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.9717995311532702, 0.9711660118330092, 0.9826981964565459, 1.0422185205277943, 1.006379483711152])
scores_mae = np.array([0.7806627125967116, 0.7653735762550717, 0.7859413084529695, 0.8231579945200965, 0.8006103038787842])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.995 ± 0.027 | MAE: 0.791 ± 0.020


### 5.3 Horizon 336

In [ ]:
!python train.py  --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_1 --random_seed 2019
!python train.py  --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_2 --random_seed 2020
!python train.py  --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_3 --random_seed 2021
!python train.py  --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_4 --random_seed 2022
!python train.py  --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_336_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=336, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_336_1_ETTh1_sl512_pl336>>>>>>>>>>>>>>>>>>>>>>>>>>
train 7793
val 2545
test 2545
Epoch: 1 cost time: 0.37962913513183594
Epoch: 1, Steps: 60 | Train Loss: 0.7390999 Vali Loss: 1.6235702 Test Loss: 1.1163625
Validation loss decreased (inf --> 1.623570).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.29624390602111816
Epoch: 2, Steps: 60 | Train Loss: 0.5746921 Vali Loss: 1.5863062 Test Loss: 1.1310408
Validation loss decreased (1.623570 --> 1.586306).  Saving m

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.9925800373679713, 1.0712574814495288, 1.1185062464914823, 1.1035649023557965, 0.993568900384401])
scores_mae = np.array([0.7966968103459007, 0.8226332664489746, 0.8609500746977957, 0.8482248406661185, 0.8132227533742001])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 1.056 ± 0.054 | MAE: 0.828 ± 0.023


### 5.4 Horizon 720

In [ ]:
!python train.py  --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_1 --random_seed 2019
!python train.py  --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_2 --random_seed 2020
!python train.py  --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_3 --random_seed 2021
!python train.py  --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_4 --random_seed 2022
!python train.py  --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_720_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=720, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_720_1_ETTh1_sl512_pl720>>>>>>>>>>>>>>>>>>>>>>>>>>
train 7409
val 2161
test 2161
Epoch: 1 cost time: 0.40407824516296387
Epoch: 1, Steps: 57 | Train Loss: 0.7480807 Vali Loss: 1.6680705 Test Loss: 1.1325370
Validation loss decreased (inf --> 1.668071).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.34616899490356445
Epoch: 2, Steps: 57 | Train Loss: 0.5918995 Vali Loss: 1.6335452 Test Loss: 1.1336009
Validation loss decreased (1.668071 --> 1.633545).  Saving m

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([1.0444950945675373, 1.0797196850180626, 1.074781022965908, 1.0495361126959324, 0.966104369610548])
scores_mae = np.array([0.8072458952665329, 0.8371136784553528, 0.8238687440752983, 0.817913893610239, 0.7905563190579414])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 1.043 ± 0.041 | MAE: 0.815 ± 0.016


## 6. Train Model on Weather

### 6.1 Horizon 96

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_96_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=96, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_96_1_custom_sl512_pl96>>>>>>>>>>>>>>>>>>>>>>>>>>
train 36280
val 5175
test 10444
Epoch: 1 cost time: 1.4689764976501465
Epoch: 1, Steps: 283 | Train Loss: 0.5538328 Vali Loss: 0.4584245 Test Loss: 0.2564672
Validation loss decreased (inf --> 0.458425).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 1.3906996250152588
Epoch: 2, Steps: 283 | Train Loss: 0.4581265 Vali Loss: 0.4565434 Test Loss: 0.2572882
Validation loss decreased (0.458425 --> 0.456543).  Saving 

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.2335099274361575, 0.20674265727952676, 0.25333500579919344, 0.2592120272693811, 0.24110162221355202])
scores_mae = np.array([0.3175220818799219, 0.29012051592638466, 0.32860518403259326, 0.33551280366049874, 0.3173807364555053])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.239 ± 0.018 | MAE: 0.318 ± 0.015


### 6.2 Horizon 192

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_192_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=192, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_192_1_custom_sl512_pl192>>>>>>>>>>>>>>>>>>>>>>>>>>
train 36184
val 5079
test 10348
Epoch: 1 cost time: 1.957441806793213
Epoch: 1, Steps: 282 | Train Loss: 0.5892556 Vali Loss: 0.5095525 Test Loss: 0.3488771
Validation loss decreased (inf --> 0.509552).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 1.4031612873077393
Epoch: 2, Steps: 282 | Train Loss: 0.4962770 Vali Loss: 0.5084364 Test Loss: 0.3408028
Validation loss decreased (0.509552 --> 0.508436).  Savi

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.2718959507532418, 0.24424777813255788, 0.26572131970897317, 0.23892522379755973, 0.26871878644451497])
scores_mae = np.array([0.3418663224205375, 0.31773060131818054, 0.33436748534440996, 0.3178629491478205, 0.3387725541368127])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.258 ± 0.014 | MAE: 0.330 ± 0.010


### 6.3 Horizon 336

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_336_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=336, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_336_1_custom_sl512_pl336>>>>>>>>>>>>>>>>>>>>>>>>>>
train 36040
val 4935
test 10204
Epoch: 1 cost time: 1.654282808303833
Epoch: 1, Steps: 281 | Train Loss: 0.6223586 Vali Loss: 0.5787166 Test Loss: 0.3681282
Validation loss decreased (inf --> 0.578717).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 1.6439592838287354
Epoch: 2, Steps: 281 | Train Loss: 0.5347956 Vali Loss: 0.5914018 Test Loss: 0.3602608
EarlyStopping counter: 1 out of 7
Updating learning rate

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.36812818267300157, 0.3412094573619999, 0.34564575807580467, 0.2976660056016113, 0.38833976093726824])
scores_mae = np.array([0.4067851293690597, 0.3852459975058519, 0.39137680281566667, 0.35817121836958055, 0.4171118057226833])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.348 ± 0.030 | MAE: 0.392 ± 0.020


### 6.4 Horizon 720

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_720_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=720, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_720_1_custom_sl512_pl720>>>>>>>>>>>>>>>>>>>>>>>>>>
train 35656
val 4551
test 9820
Epoch: 1 cost time: 2.617999792098999
Epoch: 1, Steps: 278 | Train Loss: 0.6690817 Vali Loss: 0.6960172 Test Loss: 0.4472949
Validation loss decreased (inf --> 0.696017).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 2.021291732788086
Epoch: 2, Steps: 278 | Train Loss: 0.5880245 Vali Loss: 0.6936730 Test Loss: 0.4202278
Validation loss decreased (0.696017 --> 0.693673).  Saving

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.4202278272101754, 0.4693720534835991, 0.47432004876042666, 0.4510502742701455, 0.4586101795889829])
scores_mae = np.array([0.4407422444538066, 0.46754740806002365, 0.47008430448017624, 0.45301514079696253, 0.45031428297883586])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.455 ± 0.019 | MAE: 0.456 ± 0.011


## 7. Train Model on ETTm1

### 7.1 Horizon 96

In [ ]:
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_ettm1_96_1 --random_seed 2019
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_ettm1_96_2 --random_seed 2020
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_ettm1_96_3 --random_seed 2021
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_ettm1_96_4 --random_seed 2022
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_ettm1_96_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_ettm1_96_1', data='ETTm1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTm1.csv', features='M', target='OT', freq='15min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=96, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_ettm1_96_1_ETTm1_sl512_pl96>>>>>>>>>>>>>>>>>>>>>>>>>>
train 33953
val 11425
test 11425
Epoch: 1 cost time: 2.912923812866211
Epoch: 1, Steps: 265 | Train Loss: 0.5223323 Vali Loss: 1.0109444 Test Loss: 0.8514198
Validation loss decreased (inf --> 1.010944).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 2.168491840362549
Epoch: 2, Steps: 265 | Train Loss: 0.3984346 Vali Loss: 0.9323225 Test Loss: 0.7851180
Validation loss decreased (1.010944 --> 0.932322).  Saving 

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.7851180310329694, 0.6961001934630148, 0.7942592218685686, 0.7054113928186759, 0.701706546913372])
scores_mae = np.array([0.6219605749912476, 0.6083708891038144, 0.6599343181326148, 0.6008927658032835, 0.6107055009081123])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.737 ± 0.044 | MAE: 0.620 ± 0.021


### 7.2 Horizon 192

In [ ]:
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_ettm1_192_1 --random_seed 2019
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_ettm1_192_2 --random_seed 2020
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_ettm1_192_3 --random_seed 2021
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_ettm1_192_4 --random_seed 2022
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_ettm1_192_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_ettm1_192_1', data='ETTm1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTm1.csv', features='M', target='OT', freq='15min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=192, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_ettm1_192_1_ETTm1_sl512_pl192>>>>>>>>>>>>>>>>>>>>>>>>>>
train 33857
val 11329
test 11329
Epoch: 1 cost time: 2.442354202270508
Epoch: 1, Steps: 264 | Train Loss: 0.5384880 Vali Loss: 1.0533338 Test Loss: 0.8679258
Validation loss decreased (inf --> 1.053334).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 2.413710832595825
Epoch: 2, Steps: 264 | Train Loss: 0.4331063 Vali Loss: 0.9695922 Test Loss: 0.8204693
Validation loss decreased (1.053334 --> 0.969592).  Sav

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.7601989003067667, 0.7770055474882777, 0.7630591111427004, 0.7923392318189144, 0.7873113690452143])
scores_mae = np.array([0.6392357925122435, 0.655352154915983, 0.6363914998417551, 0.6681546531617641, 0.6369340626353567])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.776 ± 0.013 | MAE: 0.647 ± 0.013


### 7.3 Horizon 336

In [ ]:
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_ettm1_336_1 --random_seed 2019
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_ettm1_336_2 --random_seed 2020
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_ettm1_336_3 --random_seed 2021
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_ettm1_336_4 --random_seed 2022
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_ettm1_336_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_ettm1_336_1', data='ETTm1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTm1.csv', features='M', target='OT', freq='15min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=336, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_ettm1_336_1_ETTm1_sl512_pl336>>>>>>>>>>>>>>>>>>>>>>>>>>
train 33713
val 11185
test 11185
Epoch: 1 cost time: 2.6909236907958984
Epoch: 1, Steps: 263 | Train Loss: 0.5652603 Vali Loss: 1.1069071 Test Loss: 0.9603166
Validation loss decreased (inf --> 1.106907).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 2.4511280059814453
Epoch: 2, Steps: 263 | Train Loss: 0.4643394 Vali Loss: 1.0467569 Test Loss: 0.8943152
Validation loss decreased (1.106907 --> 1.046757).  S

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.8943151954946846, 0.8141445198963428, 0.8430822741711277, 0.8462889996068231, 0.9549541956391828])
scores_mae = np.array([0.7180421516813081, 0.6654870530654644, 0.6828635869355038, 0.6847191035062418, 0.7508723955044802])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.871 ± 0.049 | MAE: 0.700 ± 0.030


### 7.4 Horizon 720

In [ ]:
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_ettm1_720_1 --random_seed 2019
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_ettm1_720_2 --random_seed 2020
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_ettm1_720_3 --random_seed 2021
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_ettm1_720_4 --random_seed 2022
!python train.py  --data ETTm1 --data_path ETTm1.csv --freq 15min --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_ettm1_720_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_ettm1_720_1', data='ETTm1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTm1.csv', features='M', target='OT', freq='15min', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=720, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_ettm1_720_1_ETTm1_sl512_pl720>>>>>>>>>>>>>>>>>>>>>>>>>>
train 33329
val 10801
test 10801
Epoch: 1 cost time: 2.974613666534424
Epoch: 1, Steps: 260 | Train Loss: 0.5899733 Vali Loss: 1.3214362 Test Loss: 1.0204071
Validation loss decreased (inf --> 1.321436).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 2.7736613750457764
Epoch: 2, Steps: 260 | Train Loss: 0.4890825 Vali Loss: 1.3011771 Test Loss: 0.9626624
Validation loss decreased (1.321436 --> 1.301177).  Sa

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.9381011745759419, 0.9219193699814024, 0.916194862198262, 1.0249019854125523, 0.9826275117340542])
scores_mae = np.array([0.7529825050206411, 0.7329375367789042, 0.7339836763484138, 0.7931305191346577, 0.7674465108485449])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.957 ± 0.041 | MAE: 0.756 ± 0.023


## 8. Train Model on Electricity

### 8.1 Horizon 96

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 96 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_96_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 96 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_96_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 96 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_96_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 96 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_96_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 96 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_96_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_electricity_96_1', data='custom', root_path='/data/datasets/public/electricity/', data_path='electricity.csv', features='M', target='None', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=96, enc_in=321, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_electricity_96_1_custom_sl512_pl96>>>>>>>>>>>>>>>>>>>>>>>>>>
train 17805
val 2537
test 5165
Epoch: 1 cost time: 14.671037197113037
Epoch: 1, Steps: 139 | Train Loss: 0.5946020 Vali Loss: 0.3927044 Test Loss: 0.4501998
Validation loss decreased (inf --> 0.392704).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 13.906782150268555
Epoch: 2, Steps: 139 | Train Loss: 0.4043683 Vali Loss: 0.3740835 Test Loss: 0.4298435
Validation loss decreased (0.392704 --> 0.37408

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.3923552118241787, 0.42551250010728836, 0.4194603770971298, 0.41836494356393816, 0.39547486528754233])
scores_mae = np.array([0.45320463329553606, 0.47770671024918554, 0.476362394541502, 0.4721073962748051, 0.4602499432861805])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.410 ± 0.014 | MAE: 0.468 ± 0.010


### 8.2 Horizon 192

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 192 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_192_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 192 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_192_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 192 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_192_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 192 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_192_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 192 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_192_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_electricity_192_1', data='custom', root_path='/data/datasets/public/electricity/', data_path='electricity.csv', features='M', target='None', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=192, enc_in=321, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_electricity_192_1_custom_sl512_pl192>>>>>>>>>>>>>>>>>>>>>>>>>>
train 17709
val 2441
test 5069
Epoch: 1 cost time: 15.720542192459106
Epoch: 1, Steps: 138 | Train Loss: 0.6254248 Vali Loss: 0.4050968 Test Loss: 0.4558499
Validation loss decreased (inf --> 0.405097).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 14.964725732803345
Epoch: 2, Steps: 138 | Train Loss: 0.4232694 Vali Loss: 0.3907778 Test Loss: 0.4294150
Validation loss decreased (0.405097 --> 0.3

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.40811401223525023, 0.39364519027563244, 0.41994773271756297, 0.43202082621745574, 0.39222165789359653])
scores_mae = np.array([0.46552479649201417, 0.4603901253296779, 0.4792312490634429, 0.4850934208967747, 0.45580206773219967])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.409 ± 0.015 | MAE: 0.469 ± 0.011


### 8.3 Horizon 336

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 336 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_336_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 336 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_336_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 336 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_336_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 336 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_336_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 336 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_336_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_electricity_336_1', data='custom', root_path='/data/datasets/public/electricity/', data_path='electricity.csv', features='M', target='None', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=336, enc_in=321, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_electricity_336_1_custom_sl512_pl336>>>>>>>>>>>>>>>>>>>>>>>>>>
train 17565
val 2297
test 4925
Epoch: 1 cost time: 18.120405197143555
Epoch: 1, Steps: 137 | Train Loss: 0.5884905 Vali Loss: 0.3882942 Test Loss: 0.4374748
Validation loss decreased (inf --> 0.388294).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 18.020657300949097
Epoch: 2, Steps: 137 | Train Loss: 0.3998735 Vali Loss: 0.3714277 Test Loss: 0.4164767
Validation loss decreased (0.388294 --> 0.3

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.4029179372285542, 0.43214398230377, 0.40636623376294184, 0.39823065462865326, 0.38915375267204483])
scores_mae = np.array([0.4680774141299097, 0.486111160955931, 0.4649997910386638, 0.46131987007040726, 0.4594380753604989])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.406 ± 0.014 | MAE: 0.468 ± 0.010


### 8.4 Horizon 720

In [ ]:
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 720 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_720_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 720 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_720_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 720 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_720_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 720 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_720_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/electricity/ --data_path electricity.csv --target None --pred_len 720 --enc_in 321 --train_epochs 50 --patience 7 --model_id lstm_electricity_720_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_electricity_720_1', data='custom', root_path='/data/datasets/public/electricity/', data_path='electricity.csv', features='M', target='None', freq='h', percent=100, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=720, enc_in=321, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_electricity_720_1_custom_sl512_pl720>>>>>>>>>>>>>>>>>>>>>>>>>>
train 17181
val 1913
test 4541
Epoch: 1 cost time: 24.990111827850342
Epoch: 1, Steps: 134 | Train Loss: 0.6075261 Vali Loss: 0.4070031 Test Loss: 0.4516748
Validation loss decreased (inf --> 0.407003).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 25.814889669418335
Epoch: 2, Steps: 134 | Train Loss: 0.4120432 Vali Loss: 0.3820111 Test Loss: 0.4283910
Validation loss decreased (0.407003 --> 0.3

In [ ]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.40506829960005625, 0.4024515526635306, 0.427509754044669, 0.4050204600606646, 0.4205167361668178])
scores_mae = np.array([0.4669055368219103, 0.4630160561629704, 0.48789187244006565, 0.4648182783808027, 0.477515024798257])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.412 ± 0.010 | MAE: 0.472 ± 0.009


## 9. Train Model on ETTh1 (10% data)

### 9.1 Horizon 96

In [8]:
!python train.py  --percent 10 --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_1 --random_seed 2019
!python train.py  --percent 10 --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_2 --random_seed 2020
!python train.py  --percent 10 --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_3 --random_seed 2021
!python train.py  --percent 10 --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_4 --random_seed 2022
!python train.py  --percent 10 --pred_len 96 --train_epochs 50 --patience 7 --model_id lstm_etth1_96_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_96_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=96, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_96_1_ETTh1_sl512_pl96>>>>>>>>>>>>>>>>>>>>>>>>>>
train 717
val 2785
test 2785
Epoch: 1 cost time: 0.7784006595611572
Epoch: 1, Steps: 5 | Train Loss: 1.9582138 Vali Loss: 1.5400177 Test Loss: 1.1610594
Validation loss decreased (inf --> 1.540018).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.12926816940307617
Epoch: 2, Steps: 5 | Train Loss: 1.6169693 Vali Loss: 1.5370130 Test Loss: 1.1724048
Validation loss decreased (1.540018 --> 1.537013).  Saving model ...


In [16]:
# Evaluate the results
import numpy as np
scores_mse = np.array([1.1724048313640414, 1.1519003141494024, 1.1658355111167544, 1.1553462516693842, 1.167631013052804])
scores_mae = np.array([0.8038778390203204, 0.8041955800283522, 0.8073671630450657, 0.8041410446166992, 0.8133615652720133])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 1.163 ± 0.008 | MAE: 0.807 ± 0.004


### 9.2 Horizon 192

In [9]:
!python train.py  --percent 10 --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_1 --random_seed 2019
!python train.py  --percent 10 --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_2 --random_seed 2020
!python train.py  --percent 10 --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_3 --random_seed 2021
!python train.py  --percent 10 --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_4 --random_seed 2022
!python train.py  --percent 10 --pred_len 192 --train_epochs 50 --patience 7 --model_id lstm_etth1_192_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_192_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=192, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_192_1_ETTh1_sl512_pl192>>>>>>>>>>>>>>>>>>>>>>>>>>
train 621
val 2689
test 2689
Epoch: 1 cost time: 0.37302398681640625
Epoch: 1, Steps: 4 | Train Loss: 2.0012593 Vali Loss: 1.5628638 Test Loss: 1.1714350
Validation loss decreased (inf --> 1.562864).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.1058661937713623
Epoch: 2, Steps: 4 | Train Loss: 1.7468641 Vali Loss: 1.5611181 Test Loss: 1.1745480
Validation loss decreased (1.562864 --> 1.561118).  Saving model 

In [17]:
# Evaluate the results
import numpy as np
scores_mse = np.array([1.1745479589416867, 1.1581323459034873, 1.1600363765444075, 1.1685151173954917, 1.1659985241435824])
scores_mae = np.array([0.8127901582490831, 0.810175012974512, 0.8114658026468187, 0.8146916230519613, 0.8161591064362299])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 1.165 ± 0.006 | MAE: 0.813 ± 0.002


### 9.3 Horizon 336

In [10]:
!python train.py  --percent 10 --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_1 --random_seed 2019
!python train.py  --percent 10 --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_2 --random_seed 2020
!python train.py  --percent 10 --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_3 --random_seed 2021
!python train.py  --percent 10 --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_4 --random_seed 2022
!python train.py  --percent 10 --pred_len 336 --train_epochs 50 --patience 7 --model_id lstm_etth1_336_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_336_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=336, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_336_1_ETTh1_sl512_pl336>>>>>>>>>>>>>>>>>>>>>>>>>>
train 477
val 2545
test 2545
Epoch: 1 cost time: 0.363861083984375
Epoch: 1, Steps: 3 | Train Loss: 1.9551430 Vali Loss: 1.6024695 Test Loss: 1.1544138
Validation loss decreased (inf --> 1.602469).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.13010263442993164
Epoch: 2, Steps: 3 | Train Loss: 1.7847625 Vali Loss: 1.6049350 Test Loss: 1.1524088
EarlyStopping counter: 1 out of 7
Updating learning rate to 0.01
E

In [18]:
# Evaluate the results
import numpy as np
scores_mse = np.array([1.1544138444097418, 1.1490317645825838, 1.1422987172478123, 1.1512258993951898, 1.1578757762908936])
scores_mae = np.array([0.8191475397662112, 0.8135245756099099, 0.8132458925247192, 0.8130139457552057, 0.819916119700984])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 1.151 ± 0.005 | MAE: 0.816 ± 0.003


### 9.4 Horizon 720

In [11]:
!python train.py  --percent 10 --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_1 --random_seed 2019
!python train.py  --percent 10 --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_2 --random_seed 2020
!python train.py  --percent 10 --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_3 --random_seed 2021
!python train.py  --percent 10 --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_4 --random_seed 2022
!python train.py  --percent 10 --pred_len 720 --train_epochs 50 --patience 7 --model_id lstm_etth1_720_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_etth1_720_1', data='ETTh1', root_path='/data/datasets/public/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=720, enc_in=7, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_etth1_720_1_ETTh1_sl512_pl720>>>>>>>>>>>>>>>>>>>>>>>>>>
train 93
val 2161
test 2161
Epoch: 1 cost time: 0.0658266544342041
Epoch: 1, Steps: 0 | Train Loss: nan Vali Loss: 1.6552637 Test Loss: 1.1436177
Validation loss decreased (inf --> 1.655264).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.04225945472717285
Epoch: 2, Steps: 0 | Train Loss: nan Vali Loss: 1.6560336 Test Loss: 1.1436177
EarlyStopping counter: 1 out of 7
Updating learning rate to 0.01
Epoch: 3 cost

In [19]:
# Evaluate the results
import numpy as np
scores_mse = np.array([1.1436177380383015, 1.1399742476642132, 1.141637559980154, 1.1467835754156113, 1.156235545873642])
scores_mae = np.array([0.8226250931620598, 0.8205880112946033, 0.8225935474038124, 0.8225218579173088, 0.8254912011325359])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 1.146 ± 0.006 | MAE: 0.823 ± 0.002


## 10. Train Model on Weather (10% data)

### 10.1 Horizon 96

In [12]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 96 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_96_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_96_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=96, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_96_1_custom_sl512_pl96>>>>>>>>>>>>>>>>>>>>>>>>>>
train 3542
val 5175
test 10444
Epoch: 1 cost time: 0.6608130931854248
Epoch: 1, Steps: 27 | Train Loss: 0.4859642 Vali Loss: 0.8603176 Test Loss: 0.3488677
Validation loss decreased (inf --> 0.860318).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.4267611503601074
Epoch: 2, Steps: 27 | Train Loss: 0.2133152 Vali Loss: 0.8274318 Test Loss: 0.3055076
Validation loss decreased (0.860318 --> 0.827432).  Saving mode

In [20]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.2582068806629122, 0.2431039539752183, 0.2700049719876713, 0.25089416147014243, 0.24623360374459513])
scores_mae = np.array([0.3307058404625198, 0.31973407279562066, 0.33899222608701685, 0.3232549119878698, 0.3166607793098615])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.254 ± 0.010 | MAE: 0.326 ± 0.008


### 10.2 Horizon 192

In [13]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 192 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_192_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_192_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=192, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_192_1_custom_sl512_pl192>>>>>>>>>>>>>>>>>>>>>>>>>>
train 3446
val 5079
test 10348
Epoch: 1 cost time: 0.6563930511474609
Epoch: 1, Steps: 26 | Train Loss: 0.5055190 Vali Loss: 0.8820753 Test Loss: 0.3735330
Validation loss decreased (inf --> 0.882075).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.4219057559967041
Epoch: 2, Steps: 26 | Train Loss: 0.2358497 Vali Loss: 0.8500836 Test Loss: 0.3383739
Validation loss decreased (0.882075 --> 0.850084).  Saving 

In [21]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.31209797887131574, 0.34422834040597083, 0.34538237685337664, 0.3093436907045543, 0.36004262417554855])
scores_mae = np.array([0.3712325718253851, 0.3920957470312715, 0.3877372395247221, 0.36817251816391944, 0.3998246883973479])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.334 ± 0.020 | MAE: 0.384 ± 0.012


### 10.3 Horizon 336

In [14]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 336 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_336_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_336_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=336, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_336_1_custom_sl512_pl336>>>>>>>>>>>>>>>>>>>>>>>>>>
train 3302
val 4935
test 10204
Epoch: 1 cost time: 0.6952614784240723
Epoch: 1, Steps: 25 | Train Loss: 0.5283272 Vali Loss: 0.9234957 Test Loss: 0.3996285
Validation loss decreased (inf --> 0.923496).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.4837491512298584
Epoch: 2, Steps: 25 | Train Loss: 0.2615607 Vali Loss: 0.9116649 Test Loss: 0.3836639
Validation loss decreased (0.923496 --> 0.911665).  Saving 

In [22]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.3775033924398543, 0.37931688554302045, 0.37195108642306507, 0.3791111724663384, 0.3882728727369369])
scores_mae = np.array([0.40644434452811373, 0.41226945968368384, 0.39929189599013026, 0.4065379943651489, 0.41090878900847855])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.379 ± 0.005 | MAE: 0.407 ± 0.005


### 10.4 Horizon 720

In [15]:
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_1 --random_seed 2019
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_2 --random_seed 2020
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_3 --random_seed 2021
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_4 --random_seed 2022
!python train.py  --data custom --root_path /data/datasets/public/weather/ --data_path weather.csv --target None --freq 10min --percent 10 --pred_len 720 --enc_in 21 --train_epochs 50 --patience 7 --model_id lstm_weather_720_5 --random_seed 2023

Using device: cuda
Args in experiment:
Namespace(random_seed=2019, model_id='lstm_weather_720_1', data='custom', root_path='/data/datasets/public/weather/', data_path='weather.csv', features='M', target='None', freq='10min', percent=10, embed='timeF', checkpoints='./checkpoints/', seq_len=512, label_len=0, pred_len=720, enc_in=21, hidden_size=8, num_layers=1, dropout=0.1, num_workers=2, itr=1, train_epochs=50, batch_size=128, patience=7, learning_rate=0.01, lradj='type3')
>>>>>>>start training : lstm_weather_720_1_custom_sl512_pl720>>>>>>>>>>>>>>>>>>>>>>>>>>
train 2918
val 4551
test 9820
Epoch: 1 cost time: 0.7853546142578125
Epoch: 1, Steps: 22 | Train Loss: 0.5618696 Vali Loss: 0.9798333 Test Loss: 0.4328102
Validation loss decreased (inf --> 0.979833).  Saving model ...
Updating learning rate to 0.01
Epoch: 2 cost time: 0.4986131191253662
Epoch: 2, Steps: 22 | Train Loss: 0.2847272 Vali Loss: 0.9809206 Test Loss: 0.4140674
EarlyStopping counter: 1 out of 7
Updating learning rate to 

In [23]:
# Evaluate the results
import numpy as np
scores_mse = np.array([0.40946936332865763, 0.41423016688541364, 0.40859562550720413, 0.4186121997864623, 0.42930784390160914])
scores_mae = np.array([0.4100997455810246, 0.4178803955253802, 0.41320603536932093, 0.42177142202854156, 0.4233504976881178])

print(f'MSE: {np.mean(scores_mse):.3f} ± {np.std(scores_mse):.3f} | MAE: {np.mean(scores_mae):.3f} ± {np.std(scores_mae):.3f}')

MSE: 0.416 ± 0.008 | MAE: 0.417 ± 0.005
